In [ ]:
import os
import sys
import numpy as np

"""
param_grid_cox_frailty = {
    'n_knots':              [4, 7, 10],
    'kappa':                [100.0, 1000.0, 10000.0],
    'hazard':               ['Splines', 'Splines-per', 'Piecewise-per', 'Piecewise-equi', 'Weibull'],
    'rand_dist':            ['Gamma', 'LogN'],
    'nb_gh':                [20, 32],
    'nb_int':               [10, 15],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold':   [110, 115, 120, 125]
}
"""

from src.dataset_manager import DatasetManager
from src.models.cox_frailty import CoxFrailty
from src.training_manager import GGSTrainingManager

sys.path.insert(0, os.path.abspath(".."))

m_train, m_test = DatasetManager.split_dataset()

Training = GGSTrainingManager(
    model=CoxFrailty(),
    list_ids=m_train
)

In [ ]:
param_grid = {
    'distribution':         ['gamma', 'gaussian'],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold':   [110, 115, 120, 125]
}

ggs = Training.group_grid_search(
    param_grid=param_grid,
    n_folds=3,
    silence=True
)

display(Training.get_ggs_results(top_n=10))

In [ ]:
from sklearn.model_selection import GroupKFold
from src.dataset_manager import DatasetManager
from src.models.cox_frailty import CoxFrailty
import pandas as pd

m_train, m_test = DatasetManager.split_dataset()
model = CoxFrailty()
X, y_surv, y_metrics, groups = model.prepare_training_data(m_train)

gkf = GroupKFold(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_surv, groups)):
    g_fold = groups[train_idx]
    y_fold = y_surv[train_idx]
    motores = len(set(g_fold.tolist()))
    eventos = y_fold['evento'].sum()
    motores_con_evento = len(set(g_fold[y_fold['evento']].tolist()))
    print(f"Fold {fold}: {motores} motores, {motores_con_evento} con evento=1, {eventos} eventos totales, ratio={motores_con_evento/motores:.2f}")

In [3]:
import warnings
warnings.filterwarnings('ignore')

from src.mad_scaler import MADScaler
from src.models.cox_frailty import CoxFrailty
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from src.dataset_manager import DatasetManager

m_train, m_test = DatasetManager.split_dataset()
model = CoxFrailty()
X, y_surv, y_metrics, groups = model.prepare_training_data(m_train)

gkf = GroupKFold(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_surv, groups)):
    model = CoxFrailty(distribution='gamma')
    pipeline = Pipeline([('scaler', MADScaler()), ('model', model)])
    pipeline.set_output(transform='pandas')
    pipeline.fit(
        X.iloc[train_idx], y_surv[train_idx],
        model__groups=groups[train_idx]
    )
    print(f"Fold {fold}: is_fitted={model.is_fitted_}")

Training engines: 140
Test engines: 60
Fold 0: is_fitted=True
Fold 1: is_fitted=True
Fold 2: is_fitted=True
Fold 3: is_fitted=False
Fold 4: is_fitted=True


In [5]:
from src.training_manager import GGSTrainingManager

m_train, m_test = DatasetManager.split_dataset()

Training = GGSTrainingManager(
    model=CoxFrailty(),
    list_ids=m_train
)

param_grid = {
    'distribution':         ['gamma', 'gaussian'],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold':   [110, 115, 120, 125]
}

ggs = Training.group_grid_search(
    param_grid=param_grid,
    n_folds=5,
    silence=True
)

display(Training.get_ggs_results(top_n=10))

Training engines: 140
Test engines: 60
Starting Grid Search: 24 configs × 5 folds = 120 tasks


GGS progress: 100%|██████████| 24/24 [59:31<00:00, 148.80s/it] 


,distribution,confidence_threshold,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,gaussian,0.5,110,1947.686673,0.5,22.483457,39.667066,1
1,gaussian,0.7,110,1947.686673,0.5,22.483457,39.667066,1
2,gaussian,0.9,110,1947.686673,0.5,22.483457,39.667066,1
3,gamma,0.5,110,1974.741332,0.5,22.709868,39.866962,1
4,gamma,0.7,110,1974.741332,0.5,22.709868,39.866962,1
5,gamma,0.9,110,1974.741332,0.5,22.709868,39.866962,1
6,gaussian,0.5,115,3211.491184,0.5,24.791312,42.540621,1
7,gaussian,0.7,115,3211.491184,0.5,24.791312,42.540621,1
8,gaussian,0.9,115,3211.491184,0.5,24.791312,42.540621,1
9,gamma,0.5,115,3256.099294,0.5,25.037064,42.754949,1


In [4]:
import csv

path = f'data/clean/data_motor_{1}.csv'

with open(path, mode='r', encoding='utf-8') as archivo:
    # Creamos el lector
    lector = csv.reader(archivo)
    
    # next() obtiene la primera línea y avanza el puntero
    cabecera = next(lector)

print(f"Cabecera: {cabecera}")

Cabecera: ['time_in_cycles', 'op_setting_1', 'op_setting_2', 'T24', 'T30', 'T50', 'P30', 'Nf', 'Nc', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'htBleed', 'W31', 'W32', 'RUL', 'evento']
